# Shapes, then probabilities

Complete the N-gram notebook first. Run the shape section after Chapter 5 and the loss section after Chapter 6. Values are authored inputs; expected fractions and matrix cells were derived independently.

In [1]:
from pathlib import Path
import sys, json, math
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "data/part-i/ngram.json").is_file()
             and (p / "src/config/book.mjs").is_file()), None)
if root is None:
    raise FileNotFoundError("book repository boundary not found")
sys.path.insert(0, str(root / "code/part-ii"))
print("Repository fixtures found")

Repository fixtures found


In [2]:
import numpy as np
import torch
from shape_practice import run as run_shapes
shape_result = run_shapes()
ids = torch.tensor([0, 1], dtype=torch.long, device="cpu")
weights = torch.tensor([0.0, 0.0], dtype=torch.float64, device="cpu", requires_grad=True)
print(tuple(ids.shape), weights.dtype, weights.device)

{
  "input_shape": [
    2,
    2,
    2
  ],
  "weight_shape": [
    2,
    3
  ],
  "output_shape": [
    2,
    2,
    3
  ],
  "output": [
    [
      [
        2.0,
        1.0,
        -1.0
      ],
      [
        1.0,
        -1.0,
        -2.0
      ]
    ],
    [
      [
        2.0,
        2.0,
        0.0
      ],
      [
        0.0,
        -1.0,
        -1.0
      ]
    ]
  ],
  "same_shape_different_axes": true
}
(2,) torch.float64 cpu


## Equal shapes can carry different meanings

Swapping sample and position axes preserves this example's numerical shape. Inspect an entry. Padding also needs a separate real length.

In [3]:
fixture = json.loads((root / "data/part-ii/arithmetic.json").read_text())
x = np.array(fixture["shape"]["input"], dtype=np.float64)
print(x[0, 1].tolist(), x.transpose(1, 0, 2)[0, 1].tolist())
padded = np.array([[2, 2], [0, 0]], dtype=np.float64)
print("valid mean:", padded[:1].mean(axis=0).tolist())
print("incorrect storage mean:", padded.mean(axis=0).tolist())

[1.0, -1.0] [2.0, 2.0]
valid mean: [2.0, 2.0]
incorrect storage mean: [1.0, 1.0]


## Chapter 6: score to loss

Compare against 1/2,1/4,1/4; ln(8)/2; sqrt(8), and the independently expanded KL terms. Loss uses log-sum-exp so a tiny probability need not become log(0).

In [4]:
from probability_practice import run as run_probabilities
probability_result = run_probabilities()

{
  "probabilities": [
    0.5,
    0.25,
    0.25
  ],
  "target_losses_nats": [
    0.6931471805599453,
    1.3862943611198906
  ],
  "mean_loss_nats": 1.0397207708399179,
  "perplexity": 2.82842712474619,
  "shifted_probabilities": [
    0.49999999999998623,
    0.2500000000000069,
    0.2500000000000069
  ],
  "kl_q_p_nats": 0.13081203594113697,
  "kl_p_q_nats": 0.14384103622589042,
  "zero_support_kl": "infinite",
  "underflow_safe_loss": 1000.0
}


In [5]:
logits = torch.tensor([[math.log(2), 0, 0], [math.log(2), 0, 0]], dtype=torch.float64)
targets = torch.tensor([0, 1], dtype=torch.long)
losses = torch.nn.functional.cross_entropy(logits, targets, reduction="none")
print("PyTorch independent path:", losses.tolist())
assert torch.allclose(losses, torch.tensor([math.log(2), math.log(4)], dtype=torch.float64), atol=1e-12)

PyTorch independent path: [0.6931471805599453, 1.3862943611198906]
